# Phase 2 Dataset Transformer — dataset_5000_fixed.jsonl -> dataset_5000_phase2.jsonl

**CPU-only notebook. No GPU needed.** Pure data transformation: AST parsing,
deterministic tree reconstruction, Hypothesis-based function execution, and
tokenizer-based per-token weight computation. No model is trained here.

**Pipeline (Steps A, B, C from the Phase 2 plan):**

| Step | What it adds | Method |
|---|---|---|
| A | `task="skeleton"` and `task="fill"` rows | Generic AST<->JSON-tree converter, verified by exact round-trip (`ast.dump()` comparison) per example |
| B | `golden_io` field on generate/repair/refactor rows | `Hypothesis.from_type()`-style input generation, function executed, (input,output) captured |
| C | `ap_weights` + `ap_input_ids` fields | AST-depth-based per-token loss weights, aligned to CodeT5-small's own tokenizer |

**Fixed in this version:** the AST<->JSON converter now safely handles Python
literal constants that are NOT natively JSON-serializable — `Ellipsis` (`...`,
found in stub functions and type hints like `Tuple[int, ...]`), `bytes` literals
(`b'\\xff\\xd8'`, common in binary-data-handling code), and `complex` numbers.
These are wrapped/unwrapped transparently (Cell 3's `safe_value`/`restore_value`),
and a final `skeleton_is_json_safe()` guard in Cell 4 skips-and-counts (never
crashes on) any further not-yet-anticipated type. Verified with self-tests
specifically covering these exact edge cases before the full dataset runs.

**Library versions (pinned, matches what's validated elsewhere in this project):**
- `transformers==4.46.3` — the version confirmed stable for the CodeT5 family across
  every other notebook in this project (newer versions caused tied-weight/rope_scaling
  issues on other models; this pin avoids repeating that)
- `hypothesis` — latest (no known version sensitivity for this use case)
- `sentencepiece` — installed defensively, some CodeT5-family tokenizer paths need it

**Validation built into the pipeline itself, not bolted on after:**
- Step A includes self-tests on three known examples (standard control flow, an
  Ellipsis stub, a bytes literal) **before touching the real dataset** (Cell 3) —
  if any fail, stop and do not proceed, something is wrong with the converter
- Every row's skeleton/fill extraction is individually round-trip-verified AND
  JSON-safety-checked; rows that fail either check are excluded, never silently
  kept broken or corrupted
- Step C includes a length-match assertion (ap_weights must align 1:1 with
  ap_input_ids) with mismatches dropped rather than silently kept misaligned
- Cell 10 is a full audit — schema completeness, coverage percentages for every
  new field, a live re-verification of 10 random skeleton/fill pairs, and printed
  samples of every task type. **Read Cell 10's output before running Cell 11
  (download).**

**Scope limitations stated explicitly (not hidden):**
- Golden I/O only covers top-level, non-async, plain functions where every parameter
  has a type hint. Classes, methods, and async functions get `golden_io: null` — this
  is reported as a coverage percentage in the audit, not silently skipped.
- AP-weights are computed against CodeT5-small's tokenizer specifically — if you later
  train a different tokenizer's model, these weights need recomputing for that tokenizer.


In [ ]:
# CELL 0 — INSTALL (pinned versions — see notes above)
!pip install -q transformers==4.46.3 hypothesis sentencepiece


In [ ]:
# CELL 1 — IMPORTS + CONFIG
# This notebook is CPU-only. No GPU needed — it only transforms data (AST parsing,
# deterministic tree building, Hypothesis-based function execution, tokenizer calls).
# No model is loaded or trained here.

import ast, copy, gc, hashlib, inspect, json, random, re, signal, time, typing
from collections import Counter, defaultdict
from pathlib import Path
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)

WORK = Path("/content/phase2_transform")
WORK.mkdir(parents=True, exist_ok=True)
OUT_FILE = WORK / "dataset_5000_phase2.jsonl"

import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 9), "ast.unparse() requires Python 3.9+"
print("Python version OK (ast.unparse available).")



In [ ]:
# CELL 2 — UPLOAD dataset_5000_fixed.jsonl
from google.colab import files

print("Upload dataset_5000_fixed.jsonl:")
uploaded = files.upload()
src = WORK / list(uploaded.keys())[0]
src.write_bytes(list(uploaded.values())[0])

rows = []
with src.open() as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
print(f"Loaded: {len(rows)} rows")
print(f"Tasks: {dict(Counter(r['task'] for r in rows))}")
print(f"Domains: {dict(Counter(r['domain'] for r in rows).most_common())}")



## Step A — Skeleton/Fill Extraction (generic AST<->tree converter, self-tested incl. Ellipsis/bytes edge cases)


In [ ]:
# CELL 3 — GENERIC AST <-> JSON-TREE CONVERTER (Step A core logic)
#
# Design choice: instead of hand-coding a JSON schema for every Python construct
# (FunctionDef, If, While, For, Try, With, ClassDef, ...) which is large and
# error-prone, we mirror Python's OWN ast module generically via ast.iter_fields().
# This guarantees correctness for ANY valid Python construct, not just the ones
# we thought to handle, and is verified per-example by an exact round-trip check
# below (ast.dump comparison) rather than trusted blindly.
#
# SLOT placement rule (what becomes a "fill in the content" leaf vs what stays
# structural): statement-level content (Assign, AugAssign, AnnAssign, Expr) becomes
# one slot each; the test/condition of If and While, the value of Return, and the
# exception type of an except-handler become expression slots. Everything else
# (the keywords themselves: def, if, while, for, try, class, return, except, with)
# stays as real structural nodes in the tree — this is the "skeleton".
#
# JSON-safety note: a handful of Python literal constants are NOT directly JSON
# serializable — Ellipsis ("..."), bytes (b"..."), and complex numbers (3+4j).
# These can appear inside ordinary code (stub functions, slicing, type hints,
# binary data, signal processing). safe_value/restore_value wrap and unwrap them
# so json.dumps never crashes on a real-world example.

SLOT_STATEMENT_TYPES = (ast.Assign, ast.AugAssign, ast.AnnAssign, ast.Expr)

def safe_value(value):
    """Wrap non-JSON-serializable primitive constants so json.dumps never fails."""
    if value is Ellipsis:
        return {"_const": "ellipsis"}
    if isinstance(value, bytes):
        return {"_const": "bytes", "hex": value.hex()}
    if isinstance(value, complex):
        return {"_const": "complex", "real": value.real, "imag": value.imag}
    return value

def restore_value(value):
    """Inverse of safe_value — called while rebuilding the AST from JSON."""
    if isinstance(value, dict) and "_const" in value:
        if value["_const"] == "ellipsis":
            return Ellipsis
        if value["_const"] == "bytes":
            return bytes.fromhex(value["hex"])
        if value["_const"] == "complex":
            return complex(value["real"], value["imag"])
    return value

def ast_to_dict(node, slots, slot_counter):
    """Recursively convert an ast.AST node into a plain JSON-serializable dict,
    replacing designated leaf statements/expressions with {"_type": "SLOT", ...}
    markers and recording their original source text in `slots`."""
    if isinstance(node, SLOT_STATEMENT_TYPES):
        slot_counter[0] += 1
        sid = slot_counter[0]
        slots[sid] = ast.unparse(node)
        return {"_type": "SLOT", "id": sid, "kind": "stmt"}

    if isinstance(node, ast.AST):
        result = {"_type": type(node).__name__}
        for field, value in ast.iter_fields(node):
            if field == "test" and isinstance(node, (ast.If, ast.While)) and isinstance(value, ast.AST):
                slot_counter[0] += 1
                sid = slot_counter[0]
                slots[sid] = ast.unparse(value)
                result[field] = {"_type": "SLOT", "id": sid, "kind": "expr"}
            elif field == "value" and isinstance(node, ast.Return) and isinstance(value, ast.AST):
                slot_counter[0] += 1
                sid = slot_counter[0]
                slots[sid] = ast.unparse(value)
                result[field] = {"_type": "SLOT", "id": sid, "kind": "expr"}
            elif field == "type" and isinstance(node, ast.ExceptHandler) and isinstance(value, ast.AST):
                slot_counter[0] += 1
                sid = slot_counter[0]
                slots[sid] = ast.unparse(value)
                result[field] = {"_type": "SLOT", "id": sid, "kind": "expr"}
            elif isinstance(value, list):
                result[field] = [ast_to_dict(v, slots, slot_counter) if isinstance(v, ast.AST) else safe_value(v) for v in value]
            elif isinstance(value, ast.AST):
                result[field] = ast_to_dict(value, slots, slot_counter)
            else:
                result[field] = safe_value(value)
        return result
    return safe_value(node)

def dict_to_ast(d, fills):
    """Inverse of ast_to_dict: rebuild a real ast.AST tree from the JSON dict,
    substituting each SLOT marker with its filled-in source text, re-parsed."""
    if isinstance(d, dict) and d.get("_type") == "SLOT":
        key = d["id"] if d["id"] in fills else str(d["id"])
        code_str = fills[key]
        if d["kind"] == "expr":
            return ast.parse(code_str, mode="eval").body
        parsed = ast.parse(code_str, mode="exec").body
        return parsed[0] if len(parsed) == 1 else parsed
    if isinstance(d, dict) and "_const" in d:
        return restore_value(d)
    if isinstance(d, dict) and "_type" in d:
        node_cls = getattr(ast, d["_type"])
        kwargs = {}
        for k, v in d.items():
            if k == "_type":
                continue
            if isinstance(v, list):
                kwargs[k] = [dict_to_ast(item, fills) for item in v]
            elif isinstance(v, dict):
                kwargs[k] = dict_to_ast(v, fills)
            else:
                kwargs[k] = v
        return node_cls(**kwargs)
    return d

def extract_skeleton(code):
    """Parse code, extract (skeleton_tree, slot_fills). Returns None on parse failure."""
    try:
        tree = ast.parse(code)
    except Exception:
        return None, None
    slots = {}
    counter = [0]
    skeleton_tree = ast_to_dict(tree, slots, counter)
    return skeleton_tree, slots

def rebuild_code(skeleton_tree, slot_fills):
    """Deterministically turn (skeleton_tree, slot_fills) back into real Python
    source text. No token-by-token generation involved — pure reconstruction."""
    rebuilt_node = dict_to_ast(skeleton_tree, slot_fills)
    rebuilt_node = ast.fix_missing_locations(rebuilt_node)
    return ast.unparse(rebuilt_node)

def validate_roundtrip(original_code):
    """The audit check: does skeleton+fills reconstruct EXACTLY the same AST as
    the original code? Compares ast.dump() output (structural equality), which is
    stricter and more reliable than comparing source text (whitespace/formatting
    differences in unparse output don't matter for this check)."""
    skeleton_tree, slot_fills = extract_skeleton(original_code)
    if skeleton_tree is None:
        return False, None, "parse_failed"
    try:
        rebuilt_code = rebuild_code(skeleton_tree, slot_fills)
    except Exception as e:
        return False, None, f"rebuild_failed: {e}"
    try:
        orig_dump = ast.dump(ast.parse(original_code))
        rebuilt_dump = ast.dump(ast.parse(rebuilt_code))
    except Exception as e:
        return False, None, f"reparse_failed: {e}"
    if orig_dump == rebuilt_dump:
        return True, rebuilt_code, "ok"
    return False, rebuilt_code, "dump_mismatch"

def skeleton_is_json_safe(skeleton_tree):
    """Final guard used right before json.dumps() in Cell 4 — catches any
    not-yet-anticipated non-serializable type so the loop can skip that row
    instead of crashing the whole run."""
    try:
        json.dumps(skeleton_tree)
        return True
    except (TypeError, ValueError):
        return False

print("Generic AST<->dict converter ready. Round-trip validator ready.")

# Quick self-test on a known example before running on the full dataset,
# INCLUDING the Ellipsis/bytes/complex edge cases that caused the original crash.
_test_code = (
    "def binary_search(arr, target):\n"
    "    low, high = 0, len(arr) - 1\n"
    "    while low <= high:\n"
    "        mid = (low + high) // 2\n"
    "        if arr[mid] == target:\n"
    "            return mid\n"
    "        elif arr[mid] < target:\n"
    "            low = mid + 1\n"
    "        else:\n"
    "            high = mid - 1\n"
    "    return -1"
)
_edge_case_code = (
    "def stub(x) -> None:\n"
    "    ...\n"
)
_edge_case_code2 = (
    "def encode(data):\n"
    "    marker = b'\\xff\\xd8'\n"
    "    return marker + data\n"
)

for _name, _code in [("binary_search", _test_code), ("ellipsis_stub", _edge_case_code), ("bytes_literal", _edge_case_code2)]:
    _ok, _rebuilt, _status = validate_roundtrip(_code)
    _skel, _fills = extract_skeleton(_code)
    _json_safe = skeleton_is_json_safe(_skel) if _skel else False
    print(f"Self-test [{_name}]: roundtrip={_status} (passed={_ok}) | json_safe={_json_safe}")
    if not _ok or not _json_safe:
        print(f"  WARNING in self-test '{_name}' — inspect before proceeding.")
        print(f"  rebuilt: {_rebuilt}")



In [ ]:
# CELL 4 — RUN SKELETON/FILL EXTRACTION OVER THE FULL DATASET (with validation)
# Every row's target_code gets this treatment. Rows that fail the round-trip
# check, OR whose skeleton_tree contains a type json.dumps still can't handle
# (belt-and-suspenders beyond the Ellipsis/bytes/complex cases already handled
# in Cell 3), are EXCLUDED from skeleton/fill task generation and counted —
# never silently crash the whole loop, never silently corrupt the output file.

skeleton_rows = []
fill_rows = []
roundtrip_status = Counter()
serial = 0

for r in rows:
    ok, rebuilt, status = validate_roundtrip(r["target_code"])
    roundtrip_status[status] += 1
    if not ok:
        continue

    skeleton_tree, slot_fills = extract_skeleton(r["target_code"])

    if not skeleton_is_json_safe(skeleton_tree):
        roundtrip_status["json_unsafe_skipped"] += 1
        continue

    serial += 1
    skeleton_rows.append({
        "id": f"skel_{serial:05d}",
        "task": "skeleton",
        "base_id": r["id"],
        "instruction": r["instruction"],
        "skeleton_tree": skeleton_tree,
        "num_slots": len(slot_fills),
        "domain": r["domain"],
        "difficulty": r["difficulty"],
        "content_hash": hashlib.sha256(json.dumps(skeleton_tree, sort_keys=True).encode()).hexdigest(),
    })
    fill_rows.append({
        "id": f"fill_{serial:05d}",
        "task": "fill",
        "base_id": r["id"],
        "instruction": r["instruction"],
        "skeleton_tree": skeleton_tree,
        "slot_fills": {str(k): v for k, v in slot_fills.items()},
        "domain": r["domain"],
        "difficulty": r["difficulty"],
        "content_hash": hashlib.sha256(json.dumps(slot_fills, sort_keys=True).encode()).hexdigest(),
    })

print(f"Round-trip status: {dict(roundtrip_status)}")
print(f"\nSkeleton rows created: {len(skeleton_rows)}")
print(f"Fill rows created: {len(fill_rows)}")
print(f"Round-trip pass rate: {len(skeleton_rows)}/{len(rows)} ({len(skeleton_rows)/len(rows)*100:.1f}%)")

if len(skeleton_rows) / len(rows) < 0.5:
    print("\nWARNING: round-trip pass rate below 50% — inspect dump_mismatch/rebuild_failed")
    print("examples before proceeding, the AST converter may need fixes for constructs")
    print("present in this dataset that weren't in the self-test.")



## Step B — Golden I/O Capture


In [ ]:
# CELL 5 — GOLDEN I/O CAPTURE SETUP (Step B)
# Scope: only top-level, non-async, plain functions where EVERY parameter has a
# type annotation. Classes, methods (self-taking functions), and async functions
# are skipped here (golden_io stays null for them) — they would need instantiation
# or an event loop, which adds failure modes without enough payoff for this pass.
# This scope restriction is intentional and is reported in the final audit so it's
# never a silent gap.

from hypothesis import given, settings, HealthCheck
from hypothesis import strategies as st

SUPPORTED_TYPES = (int, float, str, bool, list, dict, tuple, set)

def find_eligible_function(code):
    """Return (func_name, type_hints_dict) for the first top-level plain function
    with full type annotations, or (None, None) if none qualifies."""
    try:
        tree = ast.parse(code)
    except Exception:
        return None, None
    for node in ast.iter_child_nodes(tree):
        if isinstance(node, ast.AsyncFunctionDef):
            continue
        if isinstance(node, ast.FunctionDef):
            args = [a for a in node.args.args if a.arg != "self"]
            if len(args) != len(node.args.args):
                continue  # has 'self' -> it's a method, skip
            if not args:
                continue  # no params to generate inputs for -> skip (ambiguous scope)
            if all(a.annotation is not None for a in args):
                return node.name, {a.arg: ast.unparse(a.annotation) for a in args}
    return None, None

def annotation_to_strategy(annotation_str):
    """Map a small set of common annotation strings to Hypothesis strategies.
    Returns None for anything unsupported (caller should skip that function)."""
    simple_map = {
        "int": st.integers(min_value=-1000, max_value=1000),
        "float": st.floats(allow_nan=False, allow_infinity=False, min_value=-1000, max_value=1000),
        "str": st.text(max_size=20),
        "bool": st.booleans(),
        "list": st.lists(st.integers(min_value=-100, max_value=100), max_size=10),
        "List[int]": st.lists(st.integers(min_value=-100, max_value=100), max_size=10),
        "List[float]": st.lists(st.floats(allow_nan=False, allow_infinity=False), max_size=10),
        "List[str]": st.lists(st.text(max_size=10), max_size=10),
        "dict": st.dictionaries(st.text(max_size=8), st.integers(min_value=-100, max_value=100), max_size=5),
        "Dict[str, int]": st.dictionaries(st.text(max_size=8), st.integers(min_value=-100, max_value=100), max_size=5),
    }
    return simple_map.get(annotation_str.strip())

def capture_golden_io(code, func_name, type_hints, n_samples=5, timeout_sec=3):
    """Exec the code, build Hypothesis strategies for each param, run the function
    on n_samples generated inputs, capture (input, output) or (input, exception)."""
    ns = {}
    try:
        exec(compile(code, "<golden_io>", "exec"), ns)
        func = ns.get(func_name)
        if func is None or not callable(func):
            return None
    except Exception:
        return None

    strategies = {}
    for pname, ptype in type_hints.items():
        strat = annotation_to_strategy(ptype)
        if strat is None:
            return None  # unsupported type anywhere in the signature -> skip whole function
        strategies[pname] = strat

    captured = []

    @settings(max_examples=n_samples, deadline=None, suppress_health_check=list(HealthCheck), database=None)
    @given(**strategies)
    def _runner(**kwargs):
        if len(captured) >= n_samples:
            return
        old_handler = signal.signal(signal.SIGALRM, lambda s, f: (_ for _ in ()).throw(TimeoutError()))
        signal.alarm(timeout_sec)
        try:
            result = func(**kwargs)
            captured.append({"input": kwargs, "output": result})
        except TimeoutError:
            pass
        except Exception as e:
            captured.append({"input": kwargs, "exception": type(e).__name__})
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old_handler)

    try:
        _runner()
    except Exception:
        pass

    # Keep only JSON-serializable captures (drop any exotic return types)
    clean = []
    for c in captured:
        try:
            json.dumps(c)
            clean.append(c)
        except Exception:
            continue
    return clean if clean else None

print("Golden I/O capture functions ready (scope: top-level, fully-typed, plain functions).")



In [ ]:
# CELL 6 — RUN GOLDEN I/O CAPTURE OVER THE FULL DATASET
# Applies to generate/repair/refactor rows' target_code (the verified-correct version
# in every case — for repair/refactor, target_code IS the fixed/clean ground truth).

golden_io_status = Counter()

for r in tqdm(rows, desc="golden_io"):
    fname, hints = find_eligible_function(r["target_code"])
    if fname is None:
        r["golden_io"] = None
        golden_io_status["ineligible_no_typed_function"] += 1
        continue
    try:
        captured = capture_golden_io(r["target_code"], fname, hints, n_samples=5)
    except Exception:
        captured = None
    if captured:
        r["golden_io"] = captured
        golden_io_status["captured"] += 1
    else:
        r["golden_io"] = None
        golden_io_status["capture_failed_or_unsupported_type"] += 1

print(f"Golden I/O status: {dict(golden_io_status)}")
print(f"Coverage: {golden_io_status['captured']}/{len(rows)} ({golden_io_status['captured']/len(rows)*100:.1f}%)")



## Step C — AP-Weight Precomputation


In [ ]:
# CELL 7 — AP-WEIGHT SETUP (Step C) — AST-depth-based per-token loss weights
# Uses CodeT5-small's tokenizer (the Phase 2 target model) so weights align exactly
# to how the model will actually tokenize target_code during training. Pinned to
# transformers==4.46.3 (the version validated to work with the CodeT5 family across
# this entire project — newer versions caused tied-weight/rope_scaling issues on
# other models earlier; this pin avoids re-triggering those).

from transformers import AutoTokenizer

AP_TOKENIZER_NAME = "Salesforce/codet5-small"
ap_tokenizer = AutoTokenizer.from_pretrained(AP_TOKENIZER_NAME, use_fast=True)
print(f"Loaded tokenizer: {AP_TOKENIZER_NAME}")
print(f"Is fast tokenizer (required for offset_mapping): {ap_tokenizer.is_fast}")
assert ap_tokenizer.is_fast, "Need a fast tokenizer for return_offsets_mapping=True"

def compute_char_depth_intervals(code):
    """For every AST node with location info, record (start_char, end_char, depth)
    as a flat character-offset interval. Depth 0 = module level."""
    tree = ast.parse(code)
    lines = code.split("\n")
    line_start_offset = [0] * (len(lines) + 2)
    offset = 0
    for i, line in enumerate(lines):
        line_start_offset[i + 1] = offset + len(line) + 1
        offset += len(line) + 1

    intervals = []
    def walk(node, depth=0):
        if hasattr(node, "lineno") and getattr(node, "end_lineno", None) is not None:
            start_char = line_start_offset[node.lineno - 1] + node.col_offset
            end_char = line_start_offset[node.end_lineno - 1] + node.end_col_offset
            intervals.append((start_char, end_char, depth))
        for child in ast.iter_child_nodes(node):
            walk(child, depth + 1)
    walk(tree, 0)
    return intervals

def depth_at_char(char_pos, intervals):
    """The deepest (most specific) AST node whose range contains this character."""
    max_depth = 0
    for (s, e, d) in intervals:
        if s <= char_pos < e and d > max_depth:
            max_depth = d
    return max_depth

def compute_ap_weights(code, tokenizer, decay=0.15, max_length=512):
    """Returns (input_ids, ap_weights) — same length, ready to use as a parallel
    per-token weight array during training. Special tokens (CLS/EOS/PAD, which have
    zero-width offsets) get neutral weight 1.0."""
    try:
        intervals = compute_char_depth_intervals(code)
    except Exception:
        return None, None

    enc = tokenizer(code, return_offsets_mapping=True, truncation=True, max_length=max_length)
    weights = []
    for (s, e) in enc["offset_mapping"]:
        if s == e:
            weights.append(1.0)
            continue
        mid = (s + e) // 2
        depth = depth_at_char(mid, intervals)
        weights.append(round(1.0 / (1.0 + depth * decay), 4))
    return enc["input_ids"], weights

print("AP-weight computation functions ready.")

# Self-test on the binary_search example
_test_code = (
    "def binary_search(arr, target):\n"
    "    low, high = 0, len(arr) - 1\n"
    "    while low <= high:\n"
    "        mid = (low + high) // 2\n"
    "        if arr[mid] == target:\n"
    "            return mid\n"
    "        elif arr[mid] < target:\n"
    "            low = mid + 1\n"
    "        else:\n"
    "            high = mid - 1\n"
    "    return -1"
)
_ids, _weights = compute_ap_weights(_test_code, ap_tokenizer)
print(f"\nSelf-test: {len(_ids)} tokens, {len(_weights)} weights, length match: {len(_ids) == len(_weights)}")
print(f"Sample weights: {_weights[:10]}")



In [ ]:
# CELL 8 — RUN AP-WEIGHT COMPUTATION OVER THE FULL DATASET

ap_status = Counter()

for r in tqdm(rows, desc="ap_weights"):
    input_ids, weights = compute_ap_weights(r["target_code"], ap_tokenizer)
    if input_ids is None:
        r["ap_input_ids"] = None
        r["ap_weights"] = None
        ap_status["failed"] += 1
        continue
    if len(input_ids) != len(weights):
        r["ap_input_ids"] = None
        r["ap_weights"] = None
        ap_status["length_mismatch_dropped"] += 1
        continue
    r["ap_input_ids"] = input_ids
    r["ap_weights"] = weights
    ap_status["ok"] += 1

print(f"AP-weight status: {dict(ap_status)}")
print(f"Coverage: {ap_status['ok']}/{len(rows)} ({ap_status['ok']/len(rows)*100:.1f}%)")



## Merge + Audit + Download


In [ ]:
# CELL 9 — MERGE INTO FINAL DATASET
# Original rows (generate/repair/refactor) now carry golden_io + ap_weights fields.
# New skeleton/fill rows are appended alongside them. Everything shares the same
# JSONL file so downstream training code can filter by `task` as needed.

final_dataset = list(rows) + skeleton_rows + fill_rows
random.seed(SEED)
random.shuffle(final_dataset)

print(f"Final dataset size: {len(final_dataset)}")
print(f"  Original rows (generate/repair/refactor): {len(rows)}")
print(f"  Skeleton rows (new): {len(skeleton_rows)}")
print(f"  Fill rows (new): {len(fill_rows)}")
print(f"\nTask distribution: {dict(Counter(d['task'] for d in final_dataset).most_common())}")



In [ ]:
# CELL 10 — FULL AUDIT (run this BEFORE downloading — inspect every number below)

print("=" * 70)
print("PHASE 2 DATASET — FULL AUDIT")
print("=" * 70)

print(f"\nTotal rows: {len(final_dataset)}")
print(f"\n-- Task distribution --")
for t, c in Counter(d["task"] for d in final_dataset).most_common():
    print(f"  {t}: {c} ({c/len(final_dataset)*100:.1f}%)")

print(f"\n-- Domain distribution --")
for d, c in Counter(row["domain"] for row in final_dataset).most_common():
    print(f"  {d}: {c} ({c/len(final_dataset)*100:.1f}%)")

# 1. Schema completeness check — every row must have the fields its task type needs
print(f"\n-- Schema completeness --")
schema_errors = 0
for row in final_dataset:
    if row["task"] in ("generate", "repair", "refactor"):
        required = ["id", "task", "instruction", "target_code", "domain", "difficulty"]
    elif row["task"] == "skeleton":
        required = ["id", "task", "base_id", "skeleton_tree", "num_slots", "domain"]
    elif row["task"] == "fill":
        required = ["id", "task", "base_id", "skeleton_tree", "slot_fills", "domain"]
    else:
        required = []
    missing = [k for k in required if k not in row or row[k] is None]
    if missing:
        schema_errors += 1
print(f"  Rows with missing required fields: {schema_errors}/{len(final_dataset)}")
if schema_errors > 0:
    print("  WARNING: fix schema errors before using this dataset for training.")

# 2. Round-trip validation coverage (skeleton/fill only exist for rows that passed)
print(f"\n-- Skeleton/Fill round-trip coverage --")
base_task_rows = [r for r in final_dataset if r["task"] in ("generate", "repair", "refactor")]
print(f"  Base rows: {len(base_task_rows)}")
print(f"  Skeleton/fill pairs created: {len(skeleton_rows)}")
print(f"  Round-trip pass rate: {len(skeleton_rows)}/{len(base_task_rows)} ({len(skeleton_rows)/len(base_task_rows)*100:.1f}%)")

# 3. Golden I/O coverage
golden_covered = sum(1 for r in base_task_rows if r.get("golden_io"))
print(f"\n-- Golden I/O coverage --")
print(f"  Rows with captured golden_io: {golden_covered}/{len(base_task_rows)} ({golden_covered/len(base_task_rows)*100:.1f}%)")
print(f"  (Scope: only top-level, fully-typed, plain functions — classes/methods/async excluded by design)")

# 4. AP-weights coverage + length-match validation
ap_covered = sum(1 for r in base_task_rows if r.get("ap_weights"))
ap_length_ok = sum(1 for r in base_task_rows if r.get("ap_weights") and r.get("ap_input_ids")
                    and len(r["ap_weights"]) == len(r["ap_input_ids"]))
print(f"\n-- AP-weight coverage --")
print(f"  Rows with ap_weights: {ap_covered}/{len(base_task_rows)} ({ap_covered/len(base_task_rows)*100:.1f}%)")
print(f"  Length-matched (ap_weights vs ap_input_ids): {ap_length_ok}/{ap_covered if ap_covered else 1}")
if ap_covered > 0 and ap_length_ok != ap_covered:
    print("  WARNING: some rows have mismatched ap_weights/ap_input_ids lengths — inspect before training.")

# 5. Compile check on target_code (sanity — should still be 100% from upstream)
compile_ok = 0
for r in base_task_rows:
    try:
        ast.parse(r["target_code"])
        compile_ok += 1
    except Exception:
        pass
print(f"\n-- Target code compile check --")
print(f"  Compiles: {compile_ok}/{len(base_task_rows)} ({compile_ok/len(base_task_rows)*100:.1f}%)")

# 6. Skeleton/fill self-consistency: re-run rebuild_code on a sample and verify it still works
print(f"\n-- Skeleton/Fill sample re-verification (10 random pairs) --")
sample_fill_rows = random.sample(fill_rows, min(10, len(fill_rows)))
reverify_ok = 0
for fr in sample_fill_rows:
    try:
        rebuilt = rebuild_code(fr["skeleton_tree"], fr["slot_fills"])
        ast.parse(rebuilt)
        reverify_ok += 1
    except Exception:
        pass
print(f"  Re-verified OK: {reverify_ok}/{len(sample_fill_rows)}")

# 7. Sample printout — one example per task type
print(f"\n{'='*70}")
print("SAMPLES (one per task type)")
print(f"{'='*70}")
for task in ["generate", "repair", "refactor", "skeleton", "fill"]:
    sub = [r for r in final_dataset if r["task"] == task]
    if not sub:
        continue
    s = sub[0]
    print(f"\n[{task.upper()}] id={s['id']} domain={s.get('domain')}")
    if task in ("generate", "repair", "refactor"):
        print(f"  instruction: {s['instruction'][:100]}")
        print(f"  target_code: {s['target_code'][:100]}...")
        print(f"  golden_io: {'present (' + str(len(s['golden_io'])) + ' samples)' if s.get('golden_io') else 'none (ineligible function signature)'}")
        print(f"  ap_weights: {'present (' + str(len(s['ap_weights'])) + ' tokens)' if s.get('ap_weights') else 'none'}")
    elif task == "skeleton":
        print(f"  base_id: {s['base_id']} | num_slots: {s['num_slots']}")
        print(f"  skeleton_tree keys: {list(s['skeleton_tree'].keys())[:6]}")
    elif task == "fill":
        print(f"  base_id: {s['base_id']} | slots: {list(s['slot_fills'].items())[:3]}")

print(f"\n{'='*70}")
print("AUDIT COMPLETE — review all WARNING lines above before downloading.")
print(f"{'='*70}")



In [ ]:
# CELL 11 — SAVE + DOWNLOAD
# Only run this after reviewing Cell 10's audit output and confirming there are
# no WARNING lines you're unwilling to accept.

with OUT_FILE.open("w") as f:
    for row in final_dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved: {OUT_FILE}")
print(f"Total rows: {len(final_dataset)}")
print(f"File size: {OUT_FILE.stat().st_size / 1e6:.1f} MB")

files.download(str(OUT_FILE))
print("\nDownload started.")

